# 9.6 结构化输出生成 (Structured Output Generation)

> 🕐 预估学习时间：35分钟

LLM 输出经常需要被下游程序解析（API 调用、数据抽取、函数调用），自由文本生成容易产生格式错误。本节介绍如何让模型严格输出符合预期结构的文本。

本节涵盖：
- Prompt 引导的结构化抽取
- 约束解码（Constrained Decoding）
- 文法引导生成（Grammar-Guided Generation）
- JSON Schema 约束
- 函数调用格式生成

## 1. 结构化输出概述

**为什么需要结构化输出**：
- 函数调用：模型输出需符合函数签名，才能被程序执行
- API 集成：返回 JSON 才能被前端/后端解析
- 数据抽取：从非结构化文本中提取字段
- Agent 工具调用：错误的格式会导致工具调用失败

**两种主要方法**：

1. **Prompt 引导**：通过提示词要求模型输出特定格式
   - 优点：实现简单，无需修改推理流程
   - 缺点：不保证格式正确，可能产生幻觉字段、畸形 JSON

2. **约束解码**：在每一步生成时屏蔽非法 token
   - 优点：100% 保证格式正确
   - 缺点：需要维护解析状态，可能影响内容流畅性

**核心思想**：约束解码将“格式正确”从概率问题变为确定性问题。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import math

torch.manual_seed(42)

class PromptBasedExtractor:
    """通过 prompt 模板引导模型输出结构化数据"""

    def __init__(self, schema_fields):
        self.fields = list(schema_fields)
        self.template = self._build_template()

    def _build_template(self):
        fields_str = ', '.join(self.fields)
        return (
            '请从以下文本中提取信息，输出 JSON 格式。\n'
            f'必须包含字段: {fields_str}\n'
            '只输出 JSON，不要其他内容。\n'
            '文本: {text}\n'
            'JSON:'
        )

    def mock_generate(self, text, error_mode='correct'):
        """模拟模型生成（演示 prompt 方法的常见失败模式）"""
        data = {'name': '张三', 'age': 28, 'city': '北京'}
        if error_mode == 'correct':
            return json.dumps(data, ensure_ascii=False)
        elif error_mode == 'hallucinated':
            data['email'] = 'unknown@example.com'
            return json.dumps(data, ensure_ascii=False)
        elif error_mode == 'malformed':
            # 畸形 JSON：Python dict 字符串（单引号，非法 JSON）
            return str(data)
        elif error_mode == 'extra_text':
            return '好的，结果如下： ' + json.dumps(data, ensure_ascii=False)
        return '{}'

    def extract(self, text, error_mode='correct'):
        prompt = self.template.format(text=text)
        output = self.mock_generate(text, error_mode)
        return prompt, output

extractor = PromptBasedExtractor(['name', 'age', 'city'])
text = '张三今年28岁，住在北京。'

print('=== Prompt-Based Structured Extraction ===')
print(f'Input text: {text}')
print(f'Required fields: {extractor.fields}')

modes = ['correct', 'hallucinated', 'malformed', 'extra_text']
print(f'\nFailure modes of prompt-based approach:')
for mode in modes:
    prompt, output = extractor.extract(text, mode)
    try:
        parsed = json.loads(output)
        status = '✓ valid JSON'
        extra = set(parsed.keys()) - set(extractor.fields)
        if extra:
            status += f' (hallucinated: {extra})'
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', output, re.DOTALL)
        if match:
            try:
                json.loads(match.group())
                status = '~ valid after regex'
            except json.JSONDecodeError:
                status = '✗ malformed JSON'
        else:
            status = '✗ no JSON found'
    print(f'\n  [{mode}]')
    print(f'    Output: {output}')
    print(f'    Status: {status}')

print(f'\nKey: Prompt-based extraction is simple but unreliable.')
print(f'Constrained decoding guarantees valid structure.')

## 2. 约束解码 (Constrained Decoding)

**核心思想**：在每一步生成时，根据当前已生成内容判断哪些 token 是合法的，将非法 token 的 logit 设为 $-\infty$，使模型只能从合法 token 中采样。

**FSM（有限状态机）方法**：
1. 将 JSON（或其他格式）的语法建模为状态机
2. 每生成一个 token 后，更新状态机状态
3. 根据当前状态确定允许的 token 集合
4. 屏蔽所有不允许的 token

**优势**：
- 100% 保证输出格式正确
- 不损失模型能力（只限制格式，不限制内容）
- 可与任何采样策略（greedy、top-k、top-p）配合

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import math

torch.manual_seed(42)

DQ = chr(34)  # ASCII double quote character

class JSONConstrainedDecoder:
    """字符级 JSON 约束解码器（基于有限状态机）"""

    def __init__(self, vocab):
        self.vocab = list(vocab)
        self.vocab_size = len(self.vocab)
        self.char_to_id = {c: i for i, c in enumerate(self.vocab)}

    def allowed_chars(self, state):
        """根据当前状态返回允许的字符集合"""
        states = {
            'start': {'{'},
            'key_start': {DQ, '}'},
            'in_key': set('abcdefghijklmnopqrstuvwxyz_0123456789') | {DQ},
            'colon': {':'},
            'val_start': {DQ} | set('0123456789-'),
            'in_string': set('abcdefghijklmnopqrstuvwxyz ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789') | {DQ},
            'in_number': set('0123456789') | {',', '}'},
            'after_val': {',', '}'},
            'comma': {DQ},
            'done': set(),
        }
        return states.get(state, set())

    def mask_logits(self, logits, state):
        """屏蔽非法 token 的 logits"""
        allowed = self.allowed_chars(state)
        mask = torch.full_like(logits, float('-inf'))
        for c in allowed:
            if c in self.char_to_id:
                mask[self.char_to_id[c]] = 0.0
        return logits + mask

    def transition(self, state, char):
        """状态转移"""
        if state == 'start' and char == '{':
            return 'key_start'
        if state == 'key_start':
            if char == DQ:
                return 'in_key'
            if char == '}':
                return 'done'
        if state == 'in_key':
            if char == DQ:
                return 'colon'
            return 'in_key'
        if state == 'colon' and char == ':':
            return 'val_start'
        if state == 'val_start':
            if char == DQ:
                return 'in_string'
            if char in '0123456789-':
                return 'in_number'
        if state == 'in_string':
            if char == DQ:
                return 'after_val'
            return 'in_string'
        if state == 'in_number':
            if char == ',':
                return 'comma'
            if char == '}':
                return 'done'
            return 'in_number'
        if state == 'after_val':
            if char == ',':
                return 'comma'
            if char == '}':
                return 'done'
        if state == 'comma' and char == DQ:
            return 'in_key'
        return state

    def generate(self, logits_fn, max_steps=60):
        """约束生成有效 JSON"""
        chars = []
        state = 'start'
        for _ in range(max_steps):
            if state == 'done':
                break
            context = ''.join(chars)
            logits = logits_fn(context)
            masked = self.mask_logits(logits, state)
            probs = F.softmax(masked, dim=-1)
            idx = torch.multinomial(probs, 1).item()
            char = self.vocab[idx]
            chars.append(char)
            state = self.transition(state, char)
        return ''.join(chars)

# 构建字符级词表
vocab_chars = '{}' + DQ + ':,.0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ_- '
vocab = list(vocab_chars)
decoder = JSONConstrainedDecoder(vocab)

def mock_logits(context):
    """模拟模型 logits"""
    return torch.randn(decoder.vocab_size)

print('=== JSON Constrained Decoding ===')
print(f'Vocab size: {decoder.vocab_size} characters')
print('States: start, key_start, in_key, colon, val_start, in_string, in_number, after_val, comma, done')

print('\nGenerated JSON outputs:')
for i in range(3):
    torch.manual_seed(42 + i)
    output = decoder.generate(mock_logits, max_steps=60)
    print(f'\n  Sample {i+1}: {output}')
    try:
        parsed = json.loads(output)
        print(f'  Parsed OK ✓: {parsed}')
    except json.JSONDecodeError as e:
        print(f'  Parse error: {e}')

print(f'\nKey: Constrained decoding masks invalid tokens at each step.')
print(f'Output is guaranteed to be valid JSON by construction.')

## 3. 文法引导生成 (Grammar-Guided Generation)

**核心思想**：使用上下文无关文法（CFG）定义合法输出的结构，在生成时只允许符合文法的 token。

**与约束解码的关系**：
- 约束解码是文法引导的特例（JSON 文法）
- 文法引导更通用：可定义任意结构（代码、SQL、正则表达式等）

**实现方法**：
1. 定义文法规则（如 `value -> number | string | array`）
2. 将文法转换为有限状态机（FSM）
3. 在每步生成时，根据当前 FSM 状态确定允许的 token
4. 递归处理嵌套结构（如数组中的元素）

**应用场景**：
- 代码生成：保证语法正确
- SQL 生成：防止注入
- 数据格式：CSV、XML、YAML

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re

torch.manual_seed(42)

DQ = chr(34)

class GrammarConstraint:
    """简单的上下文无关文法约束"""

    def __init__(self):
        self.rules = {
            'value': ['number', 'string', 'array'],
            'number': ['digits'],
            'string': ['DQ chars DQ'],
            'array': ['[ elements ]'],
            'elements': ['value', 'value COMMA elements'],
        }
        self.terminals = {
            'digits': set('0123456789'),
            'chars': set('abcdefghijklmnopqrstuvwxyz '),
        }

    def allowed_chars_for(self, symbol):
        """返回非终结符当前允许的字符"""
        if symbol in self.terminals:
            return self.terminals[symbol]
        if symbol == 'number':
            return set('0123456789')
        if symbol == 'string':
            return {DQ}
        if symbol == 'array':
            return {'['}
        if symbol == 'value':
            return set('0123456789') | {DQ, '['}
        return set()

class GrammarGuidedDecoder:
    """文法引导的解码器"""

    def __init__(self, vocab, grammar):
        self.vocab = list(vocab)
        self.vocab_size = len(self.vocab)
        self.char_to_id = {c: i for i, c in enumerate(self.vocab)}
        self.grammar = grammar

    def mask_logits(self, logits, allowed):
        mask = torch.full_like(logits, float('-inf'))
        for c in allowed:
            if c in self.char_to_id:
                mask[self.char_to_id[c]] = 0.0
        return logits + mask

    def generate_number(self, logits_fn, max_len=5):
        chars = []
        for _ in range(max_len):
            logits = logits_fn(''.join(chars))
            allowed = set('0123456789')
            if len(chars) > 0 and torch.rand(1).item() < 0.3:
                break
            masked = self.mask_logits(logits, allowed)
            probs = F.softmax(masked, dim=-1)
            idx = torch.multinomial(probs, 1).item()
            chars.append(self.vocab[idx])
        return ''.join(chars)

    def generate_string(self, logits_fn, max_len=8):
        chars = [DQ]
        for _ in range(max_len):
            logits = logits_fn(''.join(chars))
            allowed = set('abcdefghijklmnopqrstuvwxyz ')
            if len(chars) > 1:
                allowed.add(DQ)
            masked = self.mask_logits(logits, allowed)
            probs = F.softmax(masked, dim=-1)
            idx = torch.multinomial(probs, 1).item()
            c = self.vocab[idx]
            chars.append(c)
            if c == DQ:
                break
        if chars[-1] != DQ:
            chars.append(DQ)
        return ''.join(chars)

    def generate_array(self, logits_fn, max_elems=3):
        chars = ['[']
        for i in range(max_elems):
            if torch.rand(1).item() < 0.5:
                elem = self.generate_number(logits_fn)
            else:
                elem = self.generate_string(logits_fn)
            chars.append(elem)
            if i < max_elems - 1 and torch.rand(1).item() < 0.4:
                chars.append(',')
            else:
                break
        chars.append(']')
        return ''.join(chars)

    def generate(self, logits_fn, value_type='number'):
        if value_type == 'number':
            return self.generate_number(logits_fn)
        elif value_type == 'string':
            return self.generate_string(logits_fn)
        elif value_type == 'array':
            return self.generate_array(logits_fn)
        return ''

vocab = list('0123456789abcdefghijklmnopqrstuvwxyz[]' + DQ + ', ')
grammar = GrammarConstraint()
decoder = GrammarGuidedDecoder(vocab, grammar)

def mock_logits(context):
    return torch.randn(decoder.vocab_size)

print('=== Grammar-Guided Generation ===')
print('Grammar rules:')
for rule, prods in grammar.rules.items():
    sep = ' | '
    print(f'  {rule} -> {sep.join(prods)}')

print('\nGenerated values:')
for vtype in ['number', 'string', 'array']:
    torch.manual_seed(42)
    val = decoder.generate(mock_logits, value_type=vtype)
    print(f'  {vtype}: {val}')

print('\nGenerated arrays:')
for i in range(3):
    torch.manual_seed(42 + i)
    arr = decoder.generate(mock_logits, value_type='array')
    print(f'  array {i+1}: {arr}')

print(f'\nKey: Grammar rules define valid structures.')
print(f'The decoder samples only tokens allowed by the grammar.')

## 4. JSON Schema 约束

**JSON Schema** 是描述 JSON 数据结构的标准语言，可以指定：
- 必需字段（`required`）
- 字段类型（`type`: string, number, boolean, array, object）
- 字段描述（`description`）
- 值约束（`minimum`, `maximum`, `pattern` 等）

**约束生成流程**：
1. 解析 Schema，提取必需字段和类型
2. 按字段顺序生成，每个字段值必须符合类型约束
3. 生成完成后验证整体结构

**优势**：
- 工业标准，与 OpenAI/Anthropic 函数调用兼容
- 可表达复杂的嵌套结构
- 自动验证生成结果

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import math
import re

torch.manual_seed(42)

class JSONSchemaConstraint:
    """基于 JSON Schema 的约束生成器"""

    def __init__(self, schema):
        self.schema = schema
        self.name = schema.get('name', 'unknown')
        self.required = schema.get('required', [])
        self.properties = schema.get('properties', {})

    def generate_value(self, prop_schema):
        """根据属性 schema 生成值"""
        ptype = prop_schema.get('type', 'string')
        if ptype == 'string':
            length = torch.randint(2, 6, (1,)).item()
            chars = []
            for _ in range(length):
                idx = torch.randint(0, 26, (1,)).item()
                chars.append(chr(ord('a') + idx))
            return ''.join(chars)
        elif ptype == 'number':
            return torch.randint(1, 100, (1,)).item()
        elif ptype == 'boolean':
            return bool(torch.rand(1).item() > 0.5)
        return None

    def generate(self):
        """生成符合 schema 的 JSON"""
        result = {}
        for field in self.required:
            prop = self.properties.get(field, {})
            result[field] = self.generate_value(prop)
        return json.dumps(result, ensure_ascii=False)

    def validate(self, json_str):
        """验证 JSON 是否符合 schema"""
        try:
            obj = json.loads(json_str)
        except json.JSONDecodeError:
            return False, 'invalid JSON'
        for field in self.required:
            if field not in obj:
                return False, f'missing field: {field}'
            expected = self.properties.get(field, {}).get('type', 'string')
            actual = type(obj[field]).__name__
            type_map = {'string': 'str', 'number': 'int', 'boolean': 'bool'}
            if expected in type_map and actual != type_map[expected]:
                return False, f'{field}: expected {expected}, got {actual}'
        return True, 'valid'

# 定义不同的 JSON Schema
schemas = {
    'person': {
        'name': 'person',
        'required': ['name', 'age', 'city'],
        'properties': {
            'name': {'type': 'string'},
            'age': {'type': 'number'},
            'city': {'type': 'string'},
        }
    },
    'product': {
        'name': 'product',
        'required': ['id', 'name', 'price'],
        'properties': {
            'id': {'type': 'number'},
            'name': {'type': 'string'},
            'price': {'type': 'number'},
        }
    },
    'event': {
        'name': 'event',
        'required': ['title', 'date', 'location'],
        'properties': {
            'title': {'type': 'string'},
            'date': {'type': 'string'},
            'location': {'type': 'string'},
        }
    },
}

def mock_logits(context):
    return torch.randn(100)

print('=== JSON Schema Constrained Generation ===')
for schema_name, schema_def in schemas.items():
    constraint = JSONSchemaConstraint(schema_def)
    torch.manual_seed(42)
    generated = constraint.generate()
    valid, msg = constraint.validate(generated)
    req = constraint.required
    print(f'\n  [{schema_name}]')
    print(f'    Required fields: {req}')
    print(f'    Generated: {generated}')
    print(f'    Valid: {valid} ({msg})')

# 对比约束 vs 无约束
print('\n--- Constrained vs Unconstrained ---')
person_schema = schemas['person']
person_constraint = JSONSchemaConstraint(person_schema)

constrained_valid = 0
unconstrained_valid = 0
total = 10

for i in range(total):
    torch.manual_seed(42 + i)
    c_out = person_constraint.generate()
    c_valid, _ = person_constraint.validate(c_out)
    if c_valid:
        constrained_valid += 1

    # 无约束：随机 ASCII 字符
    torch.manual_seed(100 + i)
    length = torch.randint(5, 30, (1,)).item()
    chars = []
    for _ in range(length):
        c = chr(torch.randint(32, 127, (1,)).item())
        chars.append(c)
    u_out = ''.join(chars)
    u_valid, _ = person_constraint.validate(u_out)
    if u_valid:
        unconstrained_valid += 1

print(f'Constrained valid:   {constrained_valid}/{total}')
print(f'Unconstrained valid: {unconstrained_valid}/{total}')
print(f'\nKey: Schema constraints guarantee 100% valid output.')
print(f'Unconstrained generation almost never produces valid JSON.')

## 5. 实践：函数调用格式生成

**函数调用（Function Calling）** 是 LLM 与外部工具交互的核心机制：
1. 模型接收工具描述（名称、参数 schema）
2. 模型生成结构化的函数调用 JSON
3. 执行引擎解析 JSON 并调用对应函数
4. 将函数返回结果注入对话

**结构化输出的作用**：
- 保证生成的函数调用 JSON 格式正确
- 确保所有必需参数都存在
- 参数类型符合 schema 定义
- 消除“模型生成了无法解析的调用”的问题

**OpenAI 函数调用格式**：
- `name`: 函数名
- `arguments`: 参数对象（符合函数的 parameters schema）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import math

torch.manual_seed(42)

class FunctionCallGenerator:
    """函数调用格式生成器"""

    def __init__(self, tools):
        self.tools = {}
        for t in tools:
            name = t['name']
            self.tools[name] = t

    def generate_call(self, func_name):
        """生成符合函数签名的调用 JSON"""
        if func_name not in self.tools:
            return None
        tool = self.tools[func_name]
        params = tool.get('parameters', {})
        required = params.get('required', [])
        properties = params.get('properties', {})

        args = {}
        for field in required:
            prop = properties.get(field, {})
            ptype = prop.get('type', 'string')
            if ptype == 'string':
                length = torch.randint(2, 6, (1,)).item()
                chars = []
                for _ in range(length):
                    idx = torch.randint(0, 26, (1,)).item()
                    chars.append(chr(ord('a') + idx))
                args[field] = ''.join(chars)
            elif ptype == 'number':
                args[field] = torch.randint(1, 100, (1,)).item()
            elif ptype == 'boolean':
                args[field] = bool(torch.rand(1).item() > 0.5)

        call = {'name': func_name, 'arguments': args}
        return json.dumps(call, ensure_ascii=False)

    def validate_call(self, json_str):
        """验证函数调用格式"""
        try:
            call = json.loads(json_str)
        except json.JSONDecodeError:
            return False, 'invalid JSON'
        if 'name' not in call or 'arguments' not in call:
            return False, 'missing name or arguments'
        name = call['name']
        if name not in self.tools:
            return False, f'unknown function: {name}'
        tool = self.tools[name]
        required = tool.get('parameters', {}).get('required', [])
        args = call['arguments']
        for field in required:
            if field not in args:
                return False, f'missing argument: {field}'
        return True, 'valid'

# 定义工具 schema（OpenAI 函数调用格式）
tools = [
    {
        'name': 'get_weather',
        'description': '获取指定城市的天气',
        'parameters': {
            'type': 'object',
            'required': ['city', 'unit'],
            'properties': {
                'city': {'type': 'string', 'description': '城市名'},
                'unit': {'type': 'string', 'description': '温度单位'},
            }
        }
    },
    {
        'name': 'search_web',
        'description': '搜索网页',
        'parameters': {
            'type': 'object',
            'required': ['query'],
            'properties': {
                'query': {'type': 'string', 'description': '搜索关键词'},
                'limit': {'type': 'number', 'description': '结果数量'},
            }
        }
    },
    {
        'name': 'send_email',
        'description': '发送邮件',
        'parameters': {
            'type': 'object',
            'required': ['to', 'subject'],
            'properties': {
                'to': {'type': 'string', 'description': '收件人'},
                'subject': {'type': 'string', 'description': '主题'},
                'body': {'type': 'string', 'description': '正文'},
            }
        }
    },
]

generator = FunctionCallGenerator(tools)

print('=== Function Call Format Generation ===')
tool_names = list(generator.tools.keys())
print(f'Available tools: {tool_names}')

print('\nGenerated function calls:')
for tool in tools:
    torch.manual_seed(42)
    name = tool['name']
    call_json = generator.generate_call(name)
    valid, msg = generator.validate_call(call_json)
    req = tool['parameters']['required']
    print(f'\n  [{name}]')
    print(f'    Required args: {req}')
    print(f'    Call: {call_json}')
    print(f'    Valid: {valid} ({msg})')

# 批量生成
print('\n--- Batch Generation ---')
valid_count = 0
total = 10
for i in range(total):
    torch.manual_seed(42 + i)
    tool = tools[i % len(tools)]
    name = tool['name']
    call = generator.generate_call(name)
    v, _ = generator.validate_call(call)
    if v:
        valid_count += 1

print(f'Valid calls: {valid_count}/{total}')
print(f'\nKey: Function calling uses structured output to guarantee valid calls.')
print(f'Every generated call can be directly executed by the tool engine.')

## 📝 课后思考题

1. Prompt 引导和约束解码各自的适用场景是什么？什么情况下应该选择哪种方法？
2. 如何为嵌套的 JSON 结构（如数组中包含对象）设计状态机？
3. 约束解码会如何影响模型的生成质量和多样性？是否存在“过度约束”问题？
4. 除了 JSON，文法引导生成还可以应用在哪些场景（如代码生成、SQL、正则表达式）？